# Rung 1/4/5 — standard-scale induction run and the head-validation cascade (GPU)

Issue #169. One Kaggle/Colab session, decided **before** seeing results:

| Step | What runs | Why |
|------|-----------|-----|
| 0 | **Pilot**: 20 epochs of the standard config, timed | Measures this GPU's real throughput (and the fresh-batch data generation, which is pure Python) so the session budget is planned from a measurement instead of an estimate |
| 1 | `exp1 --standard --seeds 0,1,2` with `--stop-on-confirmed --stop-rule any` | The **emergence** measurement: when does a head first appear? Ends each seed as soon as the detector confirms one (twice in a row) |
| 2 | `exp1 --standard --seeds 0,1,2`, no early stop | The **full-budget** measurement: 3000 epochs (384k optimizer steps per seed), the budget the issue names. The model it leaves behind is the one Rungs 4 and 5 analyse |
| 3 | `exp4 --from-checkpoint <exp1's checkpoint dir>` | Activation patching + head ablation + path patching **on the very model Rung 1 trained** — one model, not two. The unit-tested-only path patching gets its first end-to-end run |
| 4 | `exp5 --activations-from <rolling checkpoint>` | The first SAE on real activations from a model with a *confirmed* head (the harvest fix landed 2026-10-12; rolling checkpoints now load) |

**Falsifiers, fixed in advance:**
- The detector is planted-tested (tests/test_planted_induction_detector.py) *before* this run: the copy offset `q-(L-1)` must fire, the previous-token offset `q-1` (what the pre-2026-10-19 detector measured and called "diag+1") must not. A "0 heads" here is a property of the model, not of untested detection code.
- If step 1 AND step 2 both find no head, that is the result: report it with manifests, do not tune until one appears.
- Every step writes `--manifest-path` under `results/`; nothing is edited by hand afterwards.
- The **content-vs-position probe** runs on every model: on this periodic task a copy-by-content circuit and a fixed-offset circuit are the same function, so only the permuted-batch probe can say which one was learned. The wording of the claim follows the probe, not the other way round.

**Before running:** enable a GPU (Kaggle: Settings -> Accelerator -> GPU T4/P100, Internet On; Colab: Runtime -> Change runtime type -> GPU). Sessions are ~9h; every long step is resumable — re-run its cell after a disconnect.

In [ ]:
BRANCH = "main"  # a pushed branch or tag — pin it so the manifest's git_sha is a real commit
!git clone --branch $BRANCH --depth 1 https://github.com/AlessioBrillo/from-gradient-to-transformer.git repo
%cd repo
!git rev-parse --short HEAD

In [ ]:
# The GPU images already ship the CUDA torch stack plus numpy, matplotlib, tqdm and
# pyyaml. `uv sync --frozen` would re-resolve the ~5GB CUDA wheel set into the
# notebook's limited working disk and die mid-download — exactly how the first
# grokking kernel errored (2026-10-14: a 64MB partial venv, torch never landed).
# Use the preinstalled stack and fill only a real gap; the manifests record the
# torch/python versions actually used.
import importlib.util
import sys

missing = [
    m
    for m in ("torch", "numpy", "matplotlib", "tqdm", "yaml")
    if importlib.util.find_spec(m) is None
]
if missing:
    !pip install -q {' '.join(missing)}

import torch

assert torch.cuda.is_available(), "no GPU — enable one before running"
print(torch.__version__, "|", torch.cuda.get_device_name(0), "| py", sys.version.split()[0])

## Step 0 — Pilot: measure this GPU before spending the session

Twenty epochs of the **standard config** (same shape, same fresh-batch regime, same detection cadence), timed. The output is the number the budget has to be planned from: the old notebook estimated "~2h" per run, which was never measured.

If the projection does not fit the session, the fix is **one seed per session** (set `SEEDS` below to a single seed), never a shorter budget: a truncated run is exactly how this repository produced under-trained false negatives.

In [ ]:
# 20 epochs of the standard config: 20 * 128 = 2560 optimizer steps.
import os, time
t0 = time.time()
!python -m src.experiments.exp1_induction_heads --vocab-size 2048 --seq-len 64 --d-model 64 --n-layers 2 --n-heads 4 --epochs 20 --batch-size 64 --num-train 8192 --batches fresh --seeds 0 --detect-every 10 --manifest-path results/probe_pilot_standard.json
pilot_seconds = time.time() - t0
steps = 20 * 128
rate = steps / max(pilot_seconds, 1e-6)
full = 3000 * 128
print(f"pilot: {pilot_seconds:.1f}s for {steps} steps -> {rate:.1f} steps/s")
print(f"projected standard run: {full / max(rate, 1e-6) / 60:.1f} min per seed")
print(f"projected 3 seeds:       {3 * full / max(rate, 1e-6) / 3600:.2f} h")

## Step 1 — The emergence measurement

How early does a head appear? `--stop-on-confirmed --stop-rule any` ends each seed as soon as the detector confirms a head twice in a row (a *first-layer* copy head counts here: it is the earliest thing that can be confirmed, and the question here is precisely *when* the first one shows up).

`SEEDS` is an environment variable so a short session can run one seed at a time; the checkpoints and the per-seed manifest are the same either way.

In [ ]:
import os
os.environ["SEEDS"] = "0,1,2"  # set to a single seed (e.g. "0") if the session is short
!python -m src.experiments.exp1_induction_heads --standard --batches fresh --seeds $SEEDS --checkpoint-dir checkpoints/induction_emergence --checkpoint-every 250 --resume --detect-every 50 --stop-on-confirmed --stop-rule any --manifest-path results/induction_standard_emergence.json

## Step 2 — The full-budget run (the measurement Rungs 4 and 5 use)

The canonical config the issue names, at its full budget: 3000 epochs = 384k optimizer steps per seed, no early stop. The model this run leaves behind is the one the circuit-patching and SAE rungs analyse.

Why a second run and not just step 1: an early stop ends the run at the *first* confirmation, and on this task the first thing the model confirms is a first-layer copy head — a different object from the two-step circuit. The full-budget run is what answers "what does the model settle into", and it is the model the cascade analyses.

In [ ]:
!python -m src.experiments.exp1_induction_heads --standard --batches fresh --seeds $SEEDS --checkpoint-dir checkpoints/induction_standard --checkpoint-every 250 --resume --detect-every 50 --manifest-path results/induction_standard.json

In [ ]:
import json

def report(path, label):
    m = json.load(open(path))
    print(f"\n=== {label} ===")
    for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
        print(
            f"seed {seed}: heads={int(metrics['total_induction_heads'])} "
            f"(L0 {int(metrics['confirmed_heads_first_layer'])}, "
            f"later {int(metrics['confirmed_heads_later_layers'])}, "
            f"mechanism {int(metrics['mechanism_confirmed'])}) | "
            f"val acc {metrics['final_val_acc']:.4f} | "
            f"induction mass {metrics['peak_induction_mass']:.3f} | "
            f"prev-token mass {metrics['peak_prev_token_mass']:.3f} | "
            f"composition {metrics['k_composition_score']:.3f} | "
            f"epochs {int(metrics['epochs_run'])} | "
            f"probe content {metrics['probe_content_mass']:.3f} vs "
            f"period {metrics['probe_period_mass']:.3f}"
        )
    return m

m_emergence = report("results/induction_standard_emergence.json", "emergence (early-stopped)")
m_full = report("results/induction_standard.json", "full budget (3000 epochs)")

FOUND_HEAD = any(int(x["total_induction_heads"]) > 0 for x in m_full["per_seed_metrics"])
CONTENT_BASED = any(float(x["probe_content_mass"]) > float(x["probe_period_mass"]) for x in m_full["per_seed_metrics"])
print(f"\nfull-budget run confirmed a head: {FOUND_HEAD}")
print(f"the copy is content-based (not a fixed offset): {CONTENT_BASED}")

In [ ]:
# Rung 4 analyses the SAME model Rung 1 trained: the checkpoint is loaded,
# not retrained. That is what makes the cascade one model instead of two.
if FOUND_HEAD:
    !python -m src.experiments.exp4_circuit_patching --from-checkpoint checkpoints/induction_standard --source-vocab-size 2048 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4 --seeds $SEEDS --manifest-path results/exp4_standard.json
else:
    print("No head confirmed at the full budget — Rung 4 has no target and is skipped.")

In [ ]:
import json

if FOUND_HEAD:
    for seed, metrics in zip(m_full["seeds"], m_full["per_seed_metrics"]):
        print(f"exp4 seed {seed}: heads={int(metrics['total_induction_heads'])} | "
              f"ablation {metrics['mean_ablation_effect']:.3f} | "
              f"path patch {metrics['mean_path_patch_effect']:.3f} | "
              f"activation recovery {metrics['mean_activation_recovery']:.3f} | "
              f"checkpoint epoch {int(metrics['checkpoint_epoch'])}")
else:
    print("No exp4 manifest: no head to target.")

# Step 4 — the SAE on real activations from a checkpoint with a confirmed head.
import glob
import os

def first_head_seed(manifest):
    if not os.path.exists(manifest):
        return None
    m = json.load(open(manifest))
    for seed, metrics in zip(m["seeds"], m["per_seed_metrics"]):
        if int(metrics["total_induction_heads"]) > 0:
            return seed
    return None

seed = first_head_seed("results/induction_standard.json")
ckpt_dir = "checkpoints/induction_standard"
if seed is None:
    # Fall back to the emergence run: that is where the head was confirmed.
    seed = first_head_seed("results/induction_standard_emergence.json")
    ckpt_dir = "checkpoints/induction_emergence"

if seed is None:
    print("No seed confirmed an induction head — the exp5 real-activation run has no")
    print("confirmed-head checkpoint to harvest from. This is the honest outcome;")
    print("the synthetic SAE baseline already reproduces.")
else:
    ckpt = os.path.join(ckpt_dir, f"exp1_checkpoint_seed{seed}.pt")
    print(f"harvesting from {ckpt}")
    !python -m src.experiments.exp5_sae_dashboard --activations-from $ckpt --source-vocab-size 2048 --source-seq-len 64 --source-d-model 64 --source-n-layers 2 --source-n-heads 4 --d-model 64 --seeds 42 --manifest-path results/exp5_real_activations.json

In [ ]:
# Collect every manifest this session produced, then download.
import glob
import os

manifests = sorted(glob.glob("results/induction_*.json") + glob.glob("results/exp4_*.json") + glob.glob("results/exp5_real_activations.json"))
manifests = [p for p in manifests if "probe_" not in p]
print("manifests:", manifests)
!zip -j induction_run_results.zip $manifests
try:
    from google.colab import files
    files.download("induction_run_results.zip")
except ImportError:
    print("Kaggle: the zip is in the notebook output (/kaggle/working).")

## After downloading

1. Unzip into a scratch dir, then pin: `python scripts/pin_remote_run.py --from <dir>` — it validates provenance (git sha must exist locally, clean tree, seeds/aggregate contract) and installs only what passes.
2. `make verify-claims` must stay green — new manifests need their `<!-- manifest: results/... -->` tags next to the numbers they back.
3. Report exactly what happened, including "no head at the full budget": the answer to issue #169 is data, either way.
4. If a head WAS confirmed: update the Rung 1 section of `portfolio/RESULTS.md` with the new numbers — the "0 / 8 heads" claim it carried was produced by a detector that measured the previous-token offset, and it is retracted by this run.
5. If the probe says the copy is **positional** rather than content-based, say so: on this task the two are indistinguishable in-distribution, and the wording in RESULTS.md and the paper has to follow the measurement.